# LeetCode 1413 — Minimum Value to Get Positive Step by Step Sum bằng Python

**Link đề:** https://leetcode.com/problems/minimum-value-to-get-positive-step-by-step-sum/
**Độ khó:** Easy · **Chủ đề:** Array, Prefix Sum

```python
class Solution:
    def minStartValue(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **prefix sum** (tổng tiền tố / tổng dồn) và `itertools.accumulate`.
2. Viết cách thử dần `startValue = 1, 2, 3, ...` (mô phỏng) và thấy giới hạn của nó.
3. Rút ra công thức: `startValue = max(1, 1 − min_prefix)` — chỉ cần **tổng dồn nhỏ nhất**.
4. Thấy tính **đơn điệu** (startValue lớn hơn thì càng an toàn) ⇒ có thể **tìm nhị phân** (binary search).
5. Trace, chứng minh, Time/Space, lỗi hay gặp (quên `max(1, ...)`), bài tập.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho mảng số nguyên `nums`. Bắt đầu với một giá trị **dương** `startValue`. Lần lượt cộng từng phần tử của `nums` (từ trái sang phải) vào; sau **mỗi** lần cộng ta có một "tổng từng bước" (step by step sum).

Trả về `startValue` **dương nhỏ nhất** sao cho tổng từng bước **không bao giờ nhỏ hơn 1**.

| `nums` | Output | Giải thích |
|---|---:|---|
| `[-3,2,-3,4,2]` | `5` | `start=4`: 1, 3, **0** ✘; `start=5`: 2, 4, 1, 5, 7 ✔ |
| `[1,2]` | `1` | tổng luôn ≥ 1 ngay cả với start nhỏ nhất `1` |
| `[1,-2,-3]` | `5` | `5+1=6, 6−2=4, 4−3=1` ✔; start=4 cho `0` ✘ |

**Constraints:** `1 <= nums.length <= 100`, `-100 <= nums[i] <= 100`.

In [1]:
examples = [
    ([-3, 2, -3, 4, 2], 5),
    ([1, 2], 1),
    ([1, -2, -3], 5),
]
for nums, expected in examples:
    print(f"nums = {nums} -> kỳ vọng {expected}")

nums = [-3, 2, -3, 4, 2] -> kỳ vọng 5
nums = [1, 2] -> kỳ vọng 1
nums = [1, -2, -3] -> kỳ vọng 5


## 2. Nền tảng Python và khái niệm

### 2.1. Mô phỏng "tổng từng bước"

Một biến `total` bắt đầu bằng `startValue`, mỗi bước `total += x`. In ra sau mỗi bước để thấy.

In [2]:
def step_sums(start: int, nums: list[int]) -> list[int]:
    total = start
    out = []
    for x in nums:
        total += x
        out.append(total)
    return out

print("start=4:", step_sums(4, [-3, 2, -3, 4, 2]))
print("start=5:", step_sums(5, [-3, 2, -3, 4, 2]))

start=4: [1, 3, 0, 4, 6]
start=5: [2, 4, 1, 5, 7]


### 2.2. Prefix sum (tổng dồn) và `itertools.accumulate`

**Prefix sum** `P[i] = nums[0] + ... + nums[i]`. Thấy ngay: tổng từng bước thứ `i` = `startValue + P[i]`.

`itertools.accumulate(nums)` sinh các tổng dồn. Nó trả về **iterator**, bọc `list(...)` để xem.

In [3]:
from itertools import accumulate

nums = [-3, 2, -3, 4, 2]
P = list(accumulate(nums))
print("prefix sums P =", P)
print("start + P     =", [5 + p for p in P])
print("step_sums(5)  =", step_sums(5, nums))

prefix sums P = [-3, -1, -4, 0, 2]
start + P     = [2, 4, 1, 5, 7]
step_sums(5)  = [2, 4, 1, 5, 7]


### 2.3. `min` và `max` của list

`min(P)` lấy giá trị nhỏ nhất; `max(1, a)` lấy số lớn hơn giữa `1` và `a` — dùng để "kẹp dưới" ở 1.

In [4]:
print("min(P) =", min(P))
for a in [-4, 0, 1, 7]:
    print(f"max(1, {a}) = {max(1, a)}")

min(P) = -4
max(1, -4) = 1
max(1, 0) = 1
max(1, 1) = 1
max(1, 7) = 7


### 2.4. `all(...)` — kiểm tra mọi phần tử thỏa điều kiện

`all(s >= 1 for s in sums)` trả `True` nếu mọi tổng ≥ 1 (dừng sớm khi gặp phần tử sai).

In [5]:
print(all(s >= 1 for s in step_sums(5, nums)))
print(all(s >= 1 for s in step_sums(4, nums)))
print(all(s >= 1 for s in []))       # list rỗng -> True (không có phản ví dụ)

True
False
True


## 3. Cách trực giác (baseline) — thử `startValue = 1, 2, 3, ...`

Bắt đầu từ `start = 1`; mô phỏng; nếu có lúc tổng < 1 thì thử `start + 1`. Lần đầu tiên hợp lệ là đáp án (vì ta thử từ nhỏ đến lớn).

In [6]:
def min_start_bruteforce(nums: list[int]) -> int:
    start = 1
    while True:
        total = start
        ok = True
        for x in nums:
            total += x
            if total < 1:
                ok = False
                break
        if ok:
            return start
        start += 1

for nums, expected in examples:
    print(nums, "->", min_start_bruteforce(nums), "| kỳ vọng", expected)

[-3, 2, -3, 4, 2] -> 5 | kỳ vọng 5
[1, 2] -> 1 | kỳ vọng 1
[1, -2, -3] -> 5 | kỳ vọng 5


`while True:` là vòng lặp "vô hạn", thoát bằng `return`. Chắc chắn dừng vì `start` đủ lớn thì mọi tổng đều ≥ 1.

### 3.1. Nút thắt

Đáp án có thể lớn tới `1 + 100 · 100 = 10 001` (100 phần tử đều `-100`). Mỗi lần thử tốn `O(n)` ⇒ tổng `O(n · answer)` ≈ `10^6` phép — vẫn chạy, nhưng **phụ thuộc vào độ lớn giá trị**, không chỉ `n`. Nếu giá trị lên `10^9` thì vô vọng.

In [7]:
tries = 0
def count_tries(nums):
    global tries
    tries = min_start_bruteforce(nums)     # số lần thử = đáp án (thử từ 1)
    return tries

worst = [-100] * 100
print("đáp án =", count_tries(worst), "-> baseline thử", tries, "giá trị start, mỗi lần duyệt", len(worst), "phần tử")

đáp án = 10001 -> baseline thử 10001 giá trị start, mỗi lần duyệt 100 phần tử


## 4. Insight — chỉ cần **tổng dồn nhỏ nhất**

Tổng bước `i` = `start + P[i]`. Điều kiện "mọi bước ≥ 1":

`start + P[i] >= 1` với mọi `i` ⇔ `start >= 1 − P[i]` với mọi `i` ⇔ `start >= 1 − min(P)`.

Thêm điều kiện `start` **dương** (`start >= 1`) ⇒

**`start = max(1, 1 − min(P))`**.

Ví dụ `[-3,2,-3,4,2]`: `P = [-3,-1,-4,0,2]`, `min = -4` ⇒ `1 − (−4) = 5` ✔.
Ví dụ `[1,2]`: `P = [1,3]`, `min = 1` ⇒ `1 − 1 = 0` ⇒ `max(1, 0) = 1` ✔.

In [8]:
for nums, expected in examples:
    P = list(accumulate(nums))
    print(f"nums={nums}: P={P}, min(P)={min(P)}, 1-min={1 - min(P)}, max(1, .)={max(1, 1 - min(P))} | kỳ vọng {expected}")

nums=[-3, 2, -3, 4, 2]: P=[-3, -1, -4, 0, 2], min(P)=-4, 1-min=5, max(1, .)=5 | kỳ vọng 5
nums=[1, 2]: P=[1, 3], min(P)=1, 1-min=0, max(1, .)=1 | kỳ vọng 1
nums=[1, -2, -3]: P=[1, -1, -4], min(P)=-4, 1-min=5, max(1, .)=5 | kỳ vọng 5


## 5. Lời giải chính — một lượt duyệt, giảng từng đoạn

Không cần lưu cả mảng `P`, chỉ cần **tổng dồn hiện tại** và **giá trị nhỏ nhất đã thấy**.

**Đoạn 1:** `prefix = 0`, `min_prefix = 0`.
  - Tại sao `min_prefix` khởi tạo `0`? Vì kết quả cuối là `max(1, 1 − min_prefix)`; khởi tạo `0` cho `1 − 0 = 1` — chính là phần "kẹp dưới ở 1". Thủ thuật này gộp luôn `max(1, ...)`.

**Đoạn 2:** duyệt `x`: `prefix += x`; `min_prefix = min(min_prefix, prefix)`.

**Đoạn 3:** trả `1 − min_prefix`.

In [9]:
def min_start_prefix(nums: list[int]) -> int:
    prefix = 0                 # Đoạn 1
    min_prefix = 0
    for x in nums:             # Đoạn 2
        prefix += x
        min_prefix = min(min_prefix, prefix)
    return 1 - min_prefix      # Đoạn 3

for nums, expected in examples:
    print(nums, "->", min_start_prefix(nums), "| kỳ vọng", expected)

[-3, 2, -3, 4, 2] -> 5 | kỳ vọng 5
[1, 2] -> 1 | kỳ vọng 1
[1, -2, -3] -> 5 | kỳ vọng 5


### 5.1. Bản một dòng với `accumulate`

`accumulate(nums, initial=0)` thêm `0` vào đầu dãy tổng dồn (Python 3.8+), tương đương việc khởi tạo `min_prefix = 0`.

In [10]:
def min_start_accumulate(nums: list[int]) -> int:
    return 1 - min(accumulate(nums, initial=0))

print(list(accumulate([1, 2], initial=0)))
print([min_start_accumulate(n) for n, _ in examples])

[0, 1, 3]
[5, 1, 5]


## 6. Cách 3 — tìm nhị phân trên `startValue`

**Tính đơn điệu:** nếu `start` hợp lệ thì `start + 1` cũng hợp lệ (mọi tổng tăng thêm 1). Vậy dãy `hợp_lệ(1), hợp_lệ(2), ...` có dạng `F F F ... F T T T ...` ⇒ tìm **vị trí T đầu tiên** bằng binary search.

Cận trên an toàn: `hi = 1 + 100 · len(nums)` (mọi phần tử ≥ −100). Mỗi lần kiểm tra `O(n)`, số lần `O(log hi)`.

Cách này chậm hơn công thức, nhưng là **khuôn mẫu quan trọng** — "binary search on answer" — dùng khi không có công thức.

In [11]:
def is_valid(start: int, nums: list[int]) -> bool:
    total = start
    for x in nums:
        total += x
        if total < 1:
            return False
    return True

def min_start_binary_search(nums: list[int]) -> int:
    lo, hi = 1, 1 + 100 * len(nums)       # hi chắc chắn hợp lệ
    while lo < hi:
        mid = (lo + hi) // 2
        if is_valid(mid, nums):
            hi = mid                      # mid hợp lệ -> đáp án <= mid
        else:
            lo = mid + 1                  # mid không hợp lệ -> đáp án > mid
    return lo

for nums, expected in examples:
    print(nums, "->", min_start_binary_search(nums), "| kỳ vọng", expected)

[-3, 2, -3, 4, 2] -> 5 | kỳ vọng 5
[1, 2] -> 1 | kỳ vọng 1
[1, -2, -3] -> 5 | kỳ vọng 5


**Bất biến binary search:** đáp án luôn nằm trong `[lo, hi]`. Khi `lo == hi` thì đó là đáp án. Dùng `hi = mid` (không phải `mid − 1`) vì `mid` **có thể** chính là đáp án.

## 7. TRACE từng bước

Trace lời giải prefix với `nums = [-3, 2, -3, 4, 2]`:

| i | x | prefix sau | min_prefix trước | min_prefix sau | ghi chú |
|---:|---:|---:|---:|---:|---|
| 0 | -3 | -3 | 0 | -3 | xuống thấp mới |
| 1 | 2 | -1 | -3 | -3 | |
| 2 | -3 | -4 | -3 | **-4** | đáy thấp nhất |
| 3 | 4 | 0 | -4 | -4 | |
| 4 | 2 | 2 | -4 | -4 | |

Kết quả `1 − (−4) = 5`. Kiểm chứng: `start = 5` → tổng thấp nhất `5 + (−4) = 1` ✔ (vừa đúng 1).

In [12]:
def trace_min_start(nums: list[int]) -> int:
    prefix, min_prefix = 0, 0
    print(f"nums = {nums}")
    for i, x in enumerate(nums):
        old_min = min_prefix
        prefix += x
        min_prefix = min(min_prefix, prefix)
        note = "đáy mới" if min_prefix < old_min else ""
        print(f"  i={i}, x={x:>3}: prefix={prefix:>3}, min_prefix {old_min:>3} -> {min_prefix:>3} {note}")
    ans = 1 - min_prefix
    print(f"=> startValue = 1 - ({min_prefix}) = {ans}; tổng từng bước: {step_sums(ans, nums)}")
    return ans

trace_min_start([-3, 2, -3, 4, 2])
print()
trace_min_start([1, 2])

nums = [-3, 2, -3, 4, 2]
  i=0, x= -3: prefix= -3, min_prefix   0 ->  -3 đáy mới
  i=1, x=  2: prefix= -1, min_prefix  -3 ->  -3 
  i=2, x= -3: prefix= -4, min_prefix  -3 ->  -4 đáy mới
  i=3, x=  4: prefix=  0, min_prefix  -4 ->  -4 
  i=4, x=  2: prefix=  2, min_prefix  -4 ->  -4 
=> startValue = 1 - (-4) = 5; tổng từng bước: [2, 4, 1, 5, 7]

nums = [1, 2]
  i=0, x=  1: prefix=  1, min_prefix   0 ->   0 
  i=1, x=  2: prefix=  3, min_prefix   0 ->   0 
=> startValue = 1 - (0) = 1; tổng từng bước: [2, 4]


1

Ví dụ `[1, 2]` cho thấy nhánh **prefix không bao giờ âm** ⇒ `min_prefix` giữ `0` ⇒ đáp án `1` — nhờ khởi tạo `min_prefix = 0`.

Trace binary search với `[-3, 2, -3, 4, 2]`:

In [13]:
def trace_binary_search(nums: list[int]) -> int:
    lo, hi = 1, 1 + 100 * len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        ok = is_valid(mid, nums)
        print(f"  lo={lo:>3}, hi={hi:>3}, mid={mid:>3}, hợp lệ? {ok!s:<5} -> ", end="")
        if ok:
            hi = mid
        else:
            lo = mid + 1
        print(f"[{lo}, {hi}]")
    print("=> đáp án", lo)
    return lo

trace_binary_search([-3, 2, -3, 4, 2])

  lo=  1, hi=501, mid=251, hợp lệ? True  -> [1, 251]
  lo=  1, hi=251, mid=126, hợp lệ? True  -> [1, 126]
  lo=  1, hi=126, mid= 63, hợp lệ? True  -> [1, 63]
  lo=  1, hi= 63, mid= 32, hợp lệ? True  -> [1, 32]
  lo=  1, hi= 32, mid= 16, hợp lệ? True  -> [1, 16]
  lo=  1, hi= 16, mid=  8, hợp lệ? True  -> [1, 8]
  lo=  1, hi=  8, mid=  4, hợp lệ? False -> [5, 8]
  lo=  5, hi=  8, mid=  6, hợp lệ? True  -> [5, 6]
  lo=  5, hi=  6, mid=  5, hợp lệ? True  -> [5, 5]
=> đáp án 5


5

`print(..., end="")` in không xuống dòng, để dòng tiếp theo nối vào.

## 8. Code nộp LeetCode

In [14]:
class Solution:
    def minStartValue(self, nums: list[int]) -> int:
        prefix = 0
        min_prefix = 0
        for x in nums:
            prefix += x
            if prefix < min_prefix:
                min_prefix = prefix
        return 1 - min_prefix

print([Solution().minStartValue(n) for n, _ in examples])

[5, 1, 5]


### Bản binary search (luyện khuôn mẫu)

In [15]:
class SolutionBinarySearch:
    def minStartValue(self, nums: list[int]) -> int:
        def ok(start):
            total = start
            for x in nums:
                total += x
                if total < 1:
                    return False
            return True

        lo, hi = 1, 1 + 100 * len(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            if ok(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

print([SolutionBinarySearch().minStartValue(n) for n, _ in examples])

[5, 1, 5]


Hàm `ok` định nghĩa **bên trong** method (nested function) và dùng được biến `nums` của method bên ngoài (closure).

### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Thử dần | start = 1, 2, ... | `O(n · ans)` | `O(1)` | phụ thuộc độ lớn giá trị |
| 2. Prefix min (1 lượt) | `1 − min(0, P)` | `O(n)` | `O(1)` | tối ưu |
| 2b. `accumulate` | như trên, 1 dòng | `O(n)` | `O(1)` (iterator) | gọn |
| 3. Binary search | đơn điệu theo start | `O(n log(n·V))` | `O(1)` | khuôn mẫu tổng quát |

`V = 100` là trị tuyệt đối lớn nhất của phần tử.

## 9. Vì sao đúng?

**Công thức.** Tổng bước `i` là `start + P[i]`. Mọi bước ≥ 1 ⇔ `start >= 1 − P[i]` ∀i ⇔ `start >= 1 − min_i P[i]`. Cộng thêm `start >= 1` ⇔ `start >= 1 − min(0, min_i P[i])`. Giá trị nhỏ nhất thỏa là dấu bằng ⇒ `1 − min(0, P[0], ..., P[n−1])`.

**Bất biến vòng lặp:** sau khi xử lý `i + 1` phần tử, `prefix = P[i]` và `min_prefix = min(0, P[0], ..., P[i])`. Hết vòng ⇒ `min_prefix` đúng như công thức.

**Tính tối thiểu:** với `start = 1 − min_prefix − 1`, hoặc `start = 0` (không dương), hoặc tồn tại `i` đạt đáy với `start + P[i] = 0 < 1` ⇒ không hợp lệ.

## 10. Time / Space

Gọi `n = len(nums)`.

- **Prefix min:** một lượt ⇒ `O(n)` thời gian, `O(1)` bộ nhớ (2 biến).
- **Binary search:** `O(log(1 + 100n))` lần kiểm tra × `O(n)` ⇒ `O(n log n)` theo constraint (≈ 14 lần với `n = 100`).
- **Thử dần:** `O(n · ans)` với `ans` có thể tới `100n + 1` ⇒ `O(n²·V)` xấu nhất.

In [16]:
import random, time
random.seed(1)
worst = [-100] * 100
for f in [min_start_bruteforce, min_start_prefix, min_start_binary_search]:
    t0 = time.perf_counter()
    for _ in range(20):
        r = f(worst)
    print(f"{f.__name__:<26} -> {r} ({1000 * (time.perf_counter() - t0):7.2f} ms / 20 lần)")

min_start_bruteforce       -> 10001 ( 363.70 ms / 20 lần)
min_start_prefix           -> 10001 (   0.09 ms / 20 lần)
min_start_binary_search    -> 10001 (   0.74 ms / 20 lần)


## 11. Bộ test

Mẫu đề; toàn dương; toàn âm cực trị; một phần tử (âm, 0, dương); đáy ở cuối; đáy ở đầu; và **đối chiếu ngẫu nhiên** mọi cách với baseline.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS -> {actual}")

sol = Solution()
check("ví dụ 1", sol.minStartValue([-3, 2, -3, 4, 2]), 5)
check("ví dụ 2", sol.minStartValue([1, 2]), 1)
check("ví dụ 3", sol.minStartValue([1, -2, -3]), 5)
check("một phần tử âm", sol.minStartValue([-5]), 6)
check("một phần tử 0", sol.minStartValue([0]), 1)
check("một phần tử dương", sol.minStartValue([100]), 1)
check("toàn -100 (n=100)", sol.minStartValue([-100] * 100), 10001)
check("đáy ở đầu", sol.minStartValue([-10, 100, 100]), 11)
check("lên rồi xuống", sol.minStartValue([5, 5, -20]), 11)

ví dụ 1                PASS -> 5
ví dụ 2                PASS -> 1
ví dụ 3                PASS -> 5
một phần tử âm         PASS -> 6
một phần tử 0          PASS -> 1
một phần tử dương      PASS -> 1
toàn -100 (n=100)      PASS -> 10001
đáy ở đầu              PASS -> 11
lên rồi xuống          PASS -> 11


In [18]:
methods = [min_start_prefix, min_start_accumulate, min_start_binary_search,
           Solution().minStartValue, SolutionBinarySearch().minStartValue]
random.seed(1413)
for _ in range(3000):
    nums = [random.randint(-100, 100) for _ in range(random.randint(1, 12))]
    expected = min_start_bruteforce(nums)
    for f in methods:
        assert f(nums) == expected, (f, nums)
    assert all(s >= 1 for s in step_sums(expected, nums))
    assert expected == 1 or not all(s >= 1 for s in step_sums(expected - 1, nums))
print("Mọi cách khớp baseline; đáp án hợp lệ và nhỏ nhất trên 3000 mảng ngẫu nhiên")

Mọi cách khớp baseline; đáp án hợp lệ và nhỏ nhất trên 3000 mảng ngẫu nhiên


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên kẹp `max(1, ...)` → trả 0 hoặc số âm

In [19]:
def wrong_no_clamp(nums):
    return 1 - min(accumulate(nums))         # SAI: không có initial=0

print("nums=[1,2]: sai:", wrong_no_clamp([1, 2]), "| đúng:", min_start_prefix([1, 2]))

nums=[1,2]: sai: 0 | đúng: 1


### Lỗi 2: Điều kiện `>= 0` thay vì `>= 1` (lệch một đơn vị)

Đề yêu cầu tổng **không nhỏ hơn 1**. Viết `- min_prefix` (quên `1 −`) cho đáp án nhỏ hơn đúng 1.

In [20]:
def wrong_off_by_one(nums):
    return max(1, -min(accumulate(nums)))    # SAI: chỉ đảm bảo tổng >= 0

print("sai:", wrong_off_by_one([-3, 2, -3, 4, 2]), "| đúng:", min_start_prefix([-3, 2, -3, 4, 2]))

sai: 4 | đúng: 5


### Lỗi 3: Chỉ nhìn **tổng cuối cùng** thay vì mọi bước

`sum(nums)` có thể dương trong khi giữa chừng tổng rất âm.

In [21]:
def wrong_final_only(nums):
    return max(1, 1 - sum(nums))             # SAI: bỏ qua các bước giữa

print("sai:", wrong_final_only([-3, 2, -3, 4, 2]), "| đúng:", min_start_prefix([-3, 2, -3, 4, 2]))

sai: 1 | đúng: 5


### Lỗi 4: Khởi tạo `min_prefix = nums[0]` rồi quên `max(1, ...)`

Nếu khởi tạo bằng phần tử đầu (thay vì `0`) thì phải nhớ kẹp `max(1, ...)`. Hoặc dùng `float('inf')` rồi kẹp. Khởi tạo `0` là gọn nhất.

In [22]:
def wrong_init(nums):
    prefix, min_prefix = 0, float("inf")
    for x in nums:
        prefix += x
        min_prefix = min(min_prefix, prefix)
    return 1 - min_prefix                    # SAI khi mọi prefix > 0

print("sai:", wrong_init([1, 2]), "| đúng:", min_start_prefix([1, 2]))

sai: 0 | đúng: 1


## 13. Mẹo nhận dạng pattern

- "Cộng dồn từ trái sang phải, điều kiện ở **mọi bước**" ⇒ nghĩ ngay **prefix sum** và **min/max của prefix**.
- "Giá trị khởi đầu nhỏ nhất để không bao giờ âm/không bao giờ dưới ngưỡng" ⇒ `ngưỡng − min_prefix` (kẹp theo điều kiện dương).
- "Tìm giá trị nhỏ nhất thỏa điều kiện" + điều kiện **đơn điệu** ⇒ có thể **binary search on answer**.
- Khởi tạo `min_prefix = 0` (tương ứng "chưa cộng gì") giúp gộp điều kiện biên.

## 14. Biến thể

1. Tổng từng bước phải **luôn ≥ k** (ngưỡng tổng quát) ⇒ `max(1, k − min_prefix)`.
2. Trả về **chỉ số** bước đạt đáy thấp nhất.
3. Bài "trạm xăng" — LeetCode 134 *Gas Station*: tìm điểm xuất phát vòng tròn sao cho bình xăng không âm; dùng cùng ý tưởng prefix min.
4. LeetCode 1480 *Running Sum of 1d Array*: chỉ tính prefix sum.

In [23]:
def min_start_threshold(nums: list[int], k: int) -> int:
    return max(1, k - min(accumulate(nums)))     # start + P[i] >= k với mọi i, và start >= 1

def lowest_step_index(nums: list[int]) -> int:
    P = list(accumulate(nums))
    return P.index(min(P))                   # vị trí đầu tiên đạt đáy

def can_complete_circuit(gas: list[int], cost: list[int]) -> int:      # LeetCode 134
    if sum(gas) < sum(cost):
        return -1
    tank, start = 0, 0
    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0:                         # không thể xuất phát từ start..i
            start, tank = i + 1, 0
    return start

print(min_start_threshold([-3, 2, -3, 4, 2], 1), min_start_threshold([-3, 2, -3, 4, 2], 10))
print("cẩn thận:", min_start_threshold([5], 3), "(start=1 đã cho 6 >= 3) - không được dùng initial=0 khi k > 1:",
      max(1, 3 - min(accumulate([5], initial=0))))
print(lowest_step_index([-3, 2, -3, 4, 2]))
print(can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]))
print(list(accumulate([1, 2, 3, 4])))     # LeetCode 1480

5 14
cẩn thận: 1 (start=1 đã cho 6 >= 3) - không được dùng initial=0 khi k > 1: 3
2
3
[1, 3, 6, 10]


Ghi chú biến thể 1: đáp án là `max(1, k − min(P))`. Mẹo `initial=0` chỉ đúng khi `k = 1` (vì khi đó "kẹp ở 1" trùng với "thêm 0 vào prefix"). Với `k > 1`, thêm `0` vào prefix sẽ ép `start >= k` một cách **sai** — dòng "cẩn thận" ở trên cho thấy `[5], k = 3` đúng là `1` chứ không phải `3`.

## 15. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** `nums = [2, -5, 1, -4, 3]`. Lập bảng prefix, tìm đáy, cho đáp án.

**Bài 2 (đoán output):** `list(accumulate([3, -1, -4]))`, `list(accumulate([3, -1], initial=0))`, `max(1, 1 - 2)`, `all([])`.

**Bài 3 (điền chỗ trống):** `exercise_min_start(nums)` — một lượt, `prefix` và `min_prefix`.

**Bài 4 (tự viết):** `exercise_min_start_bs(nums)` — binary search với hàm kiểm tra.

**Bài 5 (biến thể):** `exercise_min_start_k(nums, k)` — tổng từng bước luôn `>= k` (`k >= 1`), `startValue` dương nhỏ nhất.

**Bài 6 (mini challenge):** `exercise_best_rotation(nums)` — được phép **xoay vòng** mảng (bắt đầu từ chỉ số bất kỳ `r`, đi `nums[r], nums[r+1], ..., nums[r-1]`). Trả về `startValue` nhỏ nhất **trong mọi cách xoay**.

In [24]:
def exercise_min_start(nums: list[int]) -> int:
    # TODO 1: prefix = 0, min_prefix = 0
    # TODO 2: cộng dồn, cập nhật min
    # TODO 3: return 1 - min_prefix
    pass

def exercise_min_start_bs(nums: list[int]) -> int:
    # TODO: lo = 1, hi = 1 + 100 * len(nums)
    pass

# assert exercise_min_start([-3, 2, -3, 4, 2]) == 5
# assert exercise_min_start_bs([1, -2, -3]) == 5

In [25]:
def exercise_min_start_k(nums: list[int], k: int) -> int:
    # TODO
    pass

def exercise_best_rotation(nums: list[int]) -> int:
    # TODO
    pass

# assert exercise_min_start_k([-3, 2, -3, 4, 2], 3) == 7
# assert exercise_best_rotation([-3, 2, -3, 4, 2]) == 1

## 16. Gợi ý 3 mức

**Bài 1** — (1) Prefix: `2, -3, -2, -6, -3`. (2) Đáy `-6`. (3) `1 − (−6)`.

**Bài 3** — (1) Hai biến. (2) `min_prefix = min(min_prefix, prefix)`. (3) Khởi tạo `0` thay cho `max(1, ...)`.

**Bài 4** — (1) Viết `is_valid(start)`. (2) `mid = (lo + hi) // 2`. (3) Hợp lệ → `hi = mid`, không → `lo = mid + 1`.

**Bài 5** — (1) Điều kiện `start + P[i] >= k`. (2) `start >= k − P[i]`. (3) `max(1, k − min(P))` — **không** dùng mẹo `initial=0` khi `k > 1`.

**Bài 6** — (1) Có `n` cách xoay; với mỗi cách dùng lời giải chính. (2) `nums[r:] + nums[:r]` là mảng xoay. (3) Lấy `min` qua mọi `r` — `O(n²)` là đủ cho `n <= 100`.

## 17. Đáp án tham khảo

**Bài 1:** prefix `2, -3, -2, -6, -3`; đáy `-6` ⇒ **7**.

**Bài 2:** `[3, 2, -2]`, `[0, 3, 2]`, `1`, `True`.

In [26]:
def answer_min_start(nums):
    prefix = min_prefix = 0
    for x in nums:
        prefix += x
        min_prefix = min(min_prefix, prefix)
    return 1 - min_prefix

def answer_min_start_bs(nums):
    lo, hi = 1, 1 + 100 * len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if is_valid(mid, nums):
            hi = mid
        else:
            lo = mid + 1
    return lo

def answer_min_start_k(nums, k):
    return max(1, k - min(accumulate(nums)))

def answer_best_rotation(nums):
    return min(answer_min_start(nums[r:] + nums[:r]) for r in range(len(nums)))

assert answer_min_start([2, -5, 1, -4, 3]) == 7
assert list(accumulate([3, -1, -4])) == [3, 2, -2] and list(accumulate([3, -1], initial=0)) == [0, 3, 2]
assert max(1, 1 - 2) == 1 and all([]) is True
assert answer_min_start([-3, 2, -3, 4, 2]) == 5
assert answer_min_start_bs([1, -2, -3]) == 5
assert answer_min_start_k([-3, 2, -3, 4, 2], 3) == 7
assert answer_best_rotation([-3, 2, -3, 4, 2]) == 1
random.seed(3)
for _ in range(1000):
    nums = [random.randint(-100, 100) for _ in range(random.randint(1, 10))]
    assert answer_min_start(nums) == answer_min_start_bs(nums) == min_start_bruteforce(nums)
    k = random.randint(1, 50)
    s = answer_min_start_k(nums, k)
    assert s >= 1 and all(t >= k for t in step_sums(s, nums))
    assert s == 1 or not all(t >= k for t in step_sums(s - 1, nums))
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


Bài 6 với `[-3, 2, -3, 4, 2]`: xoay bắt đầu tại `4` → `[4, 2, -3, 2, -3]`, prefix `4, 6, 3, 5, 2` không bao giờ âm ⇒ `startValue = 1`.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def minStartValue(self, nums: list[int]) -> int:
        prefix = min_prefix = 0          # min_prefix = 0 <=> kẹp startValue >= 1
        for x in nums:
            prefix += x                  # tổng dồn
            min_prefix = min(min_prefix, prefix)
        return 1 - min_prefix            # start + đáy = 1
```

**Khung tư duy:** "điều kiện ở mọi bước của tổng dồn" ⇒ chỉ cần **bước tệ nhất** (min hoặc max prefix).

**Checklist:**
- [ ] Đáp án phải **dương** (≥ 1) — khởi tạo `min_prefix = 0` hoặc `max(1, ...)`.
- [ ] Ngưỡng là **1**, không phải 0 → `1 − min_prefix`.
- [ ] Xét **mọi bước**, không chỉ tổng cuối.
- [ ] Nếu dùng binary search: `hi` phải chắc chắn hợp lệ; `hi = mid` khi hợp lệ.